In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")


In [ ]:
tokenizer.eos_token

In [ ]:
from transformers import AutoTokenizer
import torch

# Load the Qwen 2.5 7B Instruct tokenizer
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Set the padding token to EOS, as done previously
tokenizer.pad_token = tokenizer.eos_token

#               4= 1024 - 1020
# Padding Tokens=Length of Longest Sequence in Batch−Length of Current Sequence
sentences = [
    "Hello, how are you?",  # Shorter sentence
    "The sun is shining brightly today, which is lovely." # Longer sentence
]

def print_padding_example(padding_side, sentences):
    """Tokenizes, pads, and prints results for a given padding side."""
    tokenizer.padding_side = padding_side

    # Tokenize the batch and apply padding
    output = tokenizer(sentences, padding=True, return_tensors="pt")

    # Decode the sequences back to text to visualize the padding
    decoded_sequences = [tokenizer.decode(seq, skip_special_tokens=False) for seq in output['input_ids']]

    # Get the padding token string for display
    pad_token_str = tokenizer.decode(tokenizer.pad_token_id, skip_special_tokens=False).strip()

    print(f"\n--- {padding_side.upper()} PADDING ---")

    for i, seq_id in enumerate(output['input_ids']):
        print(f"\nOriginal Text {i}: '{sentences[i]}'")
        print(f"Token IDs   {i}: {seq_id.tolist()}")
        print(f"Attention M.{i}: {output['attention_mask'][i].tolist()}")

        # This shows where the PAD token is inserted in the decoded text
        print(f"Decoded Text {i}: '{decoded_sequences[i].replace(pad_token_str, f'|{pad_token_str}|')}'")
        print(f"Padding Token: {pad_token_str} (ID: {tokenizer.pad_token_id})")

# Print the examples for both modes
print_padding_example("right", sentences)
print_padding_example("left", sentences)

In [ ]:

def format_dolly(example):


    system_content = example["instruction"]

    user_content = example["context"] if example["context"] else ""

    # Use response as the assistant message
    assistant_content = example["response"]

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    return {"text": formatted_text}

processed_dataset = dataset.map(format_dolly, remove_columns=dataset.column_names)


In [ ]:
processed_dataset

In [ ]:

def tokenize_function(examples):
    # Tokenize the 'text' column, which now holds the chat template formatted string
    return tokenizer(examples["text"], truncation=True, max_length=1024)

tokenized_dataset = processed_dataset.map(tokenize_function, batched=True)



In [ ]:
final_training_column = tokenized_dataset.remove_columns(["text", "attention_mask"])
print(final_training_column[0]["input_ids"])

In [ ]:
final_training_column = tokenized_dataset.remove_columns(["text", "attention_mask"])
print("First 10 rows of the final 'input_ids' column:")
for i in range(10):
    print(final_training_column[i]["input_ids"])

In [ ]:

# 1. Original Dataset Row
example = dataset[0]
print("--- Original Data (Example 0) ---")
print(f"Instruction (System): {example['instruction']}")
print(f"Context (User): {example['context']}")
print(f"Response (Assistant): {example['response']}")
print("-" * 40)

# 2. Chat Templated String (Re-run for display)
system_content = example["instruction"]
user_content = example["context"] if example["context"] else "None."
assistant_content = example["response"]

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": user_content},
    {"role": "assistant", "content": assistant_content},
]
formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

print("--- Chat Templated String ('text' column) ---")
print(formatted_text)
print("-" * 40)

# 3. Tokenized Output (Input IDs and Attention Mask)
input_ids = tokenized_dataset[0]["input_ids"]
attention_mask = tokenized_dataset[0]["attention_mask"] # This is where the mask is pulled from the tokenized_dataset

decoded_tokens = tokenizer.decode(input_ids)

print("--- Tokenized Output (Input IDs) ---")
print(input_ids)
print(f"Length of Input IDs: {len(input_ids)}")
print("-" * 40)

# Display the Attention Mask
print("--- Attention Mask ---")
print(attention_mask)
print(f"Length of Attention Mask: {len(attention_mask)}")
print("-" * 40)

print("--- Decoded Token String (for verification) ---")
print(decoded_tokens)